# 005 - Gold: Modelo Dimensional (Kimball)

Este notebook lê as tabelas da camada **Silver** e constrói o modelo dimensional
no schema `workspace.gold`, seguindo a metodologia de **Ralph Kimball**.

## Schema Estrela

```
              dim_tempo
             (FK_TEMPO)
                  │
                  │
dim_produto ── fato_vendas ── dim_localidade
(FK_PRODUTO)        │           (FK_LOCALIDADE)
                    │
              dim_cliente
             (FK_CLIENTE)
```

## 1. Ler tabelas da Silver

In [ ]:
df_categoria   = spark.read.format("delta").table("silver.categoria")
df_cliente     = spark.read.format("delta").table("silver.cliente")
df_endereco    = spark.read.format("delta").table("silver.endereco")
df_estado      = spark.read.format("delta").table("silver.estado")
df_fornecedor  = spark.read.format("delta").table("silver.fornecedor")
df_item_pedido = spark.read.format("delta").table("silver.item_pedido")
df_municipio   = spark.read.format("delta").table("silver.municipio")
df_pedido      = spark.read.format("delta").table("silver.pedido")
df_produto     = spark.read.format("delta").table("silver.produto")
df_regiao      = spark.read.format("delta").table("silver.regiao")

## 2. Dimensão: `dim_produto`

Desnormaliza produto + categoria + fornecedor em uma única dimensão.

In [ ]:
%%sql
DROP TABLE IF EXISTS gold.dim_produto

In [ ]:
%%sql
CREATE TABLE gold.dim_produto (
    SK_PRODUTO         BIGINT GENERATED BY DEFAULT AS IDENTITY,
    CODIGO_PRODUTO     INT,
    NOME_PRODUTO       VARCHAR(200),
    NOME_CATEGORIA     VARCHAR(100),
    NOME_FORNECEDOR    VARCHAR(200),
    VALOR_PRECO        DOUBLE
)
USING DELTA;

In [ ]:
df_produto.createOrReplaceTempView("produto")
df_categoria.createOrReplaceTempView("categoria")
df_fornecedor.createOrReplaceTempView("fornecedor")

In [ ]:
%%sql
WITH produto_relacional AS (
    SELECT p.codigo_produto,
           p.nome_produto,
           c.nome_categoria,
           f.nome_fornecedor,
           p.valor_preco
      FROM produto p
           INNER JOIN categoria c
             ON p.codigo_categoria = c.codigo_categoria
           INNER JOIN fornecedor f
             ON p.codigo_fornecedor = f.codigo_fornecedor
)
MERGE INTO gold.dim_produto AS d
USING produto_relacional AS r
ON d.codigo_produto = r.codigo_produto

WHEN MATCHED AND (d.nome_produto <> r.nome_produto OR d.nome_categoria <> r.nome_categoria
                  OR d.nome_fornecedor <> r.nome_fornecedor OR d.valor_preco <> r.valor_preco) THEN
    UPDATE SET nome_produto    = r.nome_produto,
               nome_categoria  = r.nome_categoria,
               nome_fornecedor = r.nome_fornecedor,
               valor_preco     = r.valor_preco

WHEN NOT MATCHED THEN
    INSERT (codigo_produto, nome_produto, nome_categoria, nome_fornecedor, valor_preco)
    VALUES (r.codigo_produto, r.nome_produto, r.nome_categoria, r.nome_fornecedor, r.valor_preco)

In [ ]:
%%sql
SELECT * FROM gold.dim_produto ORDER BY sk_produto

## 3. Dimensão: `dim_tempo`

Calendário gerado programaticamente de 2023 a 2026.

In [ ]:
%%sql
DROP TABLE IF EXISTS gold.dim_tempo

In [ ]:
from pyspark.sql.functions import expr, date_format

data_inicial = "2023-01-01"
data_final   = "2026-12-31"

num_dias = spark.sql(f"SELECT datediff('{data_final}', '{data_inicial}')").collect()[0][0]

df_calendario = spark.range(0, num_dias + 1) \
    .selectExpr(f"date_add(to_date('{data_inicial}'), CAST(id AS INT)) AS Data")

df_tempo = df_calendario.selectExpr(
    "Data",
    "year(Data) AS Ano",
    "month(Data) AS Mes",
    "(CASE month(Data) \
        WHEN 1  THEN 'JANEIRO'   WHEN 2  THEN 'FEVEREIRO' WHEN 3  THEN 'MARCO' \
        WHEN 4  THEN 'ABRIL'     WHEN 5  THEN 'MAIO'      WHEN 6  THEN 'JUNHO' \
        WHEN 7  THEN 'JULHO'     WHEN 8  THEN 'AGOSTO'    WHEN 9  THEN 'SETEMBRO' \
        WHEN 10 THEN 'OUTUBRO'   WHEN 11 THEN 'NOVEMBRO'  WHEN 12 THEN 'DEZEMBRO' \
    END) AS NomeMes",
    "day(Data) AS Dia",
    "(CASE dayofweek(Data) \
        WHEN 1 THEN 'DOMINGO'       WHEN 2 THEN 'SEGUNDA-FEIRA' \
        WHEN 3 THEN 'TERCA-FEIRA'   WHEN 4 THEN 'QUARTA-FEIRA' \
        WHEN 5 THEN 'QUINTA-FEIRA'  WHEN 6 THEN 'SEXTA-FEIRA' \
        WHEN 7 THEN 'SABADO' \
    END) AS NomeDiaSemana",
    "dayofweek(Data) AS NumeroDiaSemana"
)

df_tempo.display()
df_tempo.write.format("delta").mode("overwrite").saveAsTable("gold.dim_tempo")

## 4. Dimensão: `dim_cliente`

In [ ]:
%%sql
DROP TABLE IF EXISTS gold.dim_cliente

In [ ]:
%%sql
CREATE TABLE gold.dim_cliente (
    SK_CLIENTE         BIGINT GENERATED BY DEFAULT AS IDENTITY,
    CODIGO_CLIENTE     INT,
    NOME               VARCHAR(100),
    CPF                VARCHAR(11),
    SEXO               CHAR(1),
    DATA_NASCIMENTO    DATE
)
USING DELTA;

In [ ]:
df_cliente.createOrReplaceTempView("cliente")

In [ ]:
%%sql
WITH cliente_relacional AS (
    SELECT codigo_cliente,
           nome,
           cpf,
           sexo,
           data_nascimento
      FROM cliente
)
MERGE INTO gold.dim_cliente AS d
USING cliente_relacional AS r
ON r.codigo_cliente = d.codigo_cliente

WHEN MATCHED AND (r.nome <> d.nome OR r.cpf <> d.cpf OR r.sexo <> d.sexo
                  OR r.data_nascimento <> d.data_nascimento) THEN
    UPDATE SET nome            = r.nome,
               cpf             = r.cpf,
               sexo            = r.sexo,
               data_nascimento = r.data_nascimento

WHEN NOT MATCHED THEN
    INSERT (codigo_cliente, nome, cpf, sexo, data_nascimento)
    VALUES (r.codigo_cliente, r.nome, r.cpf, r.sexo, r.data_nascimento)

In [ ]:
%%sql
SELECT * FROM gold.dim_cliente ORDER BY sk_cliente

## 5. Dimensão: `dim_localidade`

In [ ]:
%%sql
DROP TABLE IF EXISTS gold.dim_localidade

In [ ]:
%%sql
CREATE TABLE gold.dim_localidade (
    SK_LOCALIDADE      BIGINT GENERATED BY DEFAULT AS IDENTITY,
    CODIGO_MUNICIPIO   INT,
    NOME_MUNICIPIO     VARCHAR(100),
    NOME_ESTADO        VARCHAR(100),
    NOME_REGIAO        VARCHAR(100)
)
USING DELTA;

In [ ]:
df_municipio.createOrReplaceTempView("municipio")
df_estado.createOrReplaceTempView("estado")
df_regiao.createOrReplaceTempView("regiao")

In [ ]:
%%sql
WITH localidade_relacional AS (
    SELECT m.codigo_municipio,
           m.nome_municipio,
           e.nome_estado,
           r.nome_regiao
      FROM municipio m
           INNER JOIN estado e ON m.codigo_estado = e.codigo_estado
           INNER JOIN regiao r ON e.codigo_regiao = r.codigo_regiao
)
MERGE INTO gold.dim_localidade AS d
USING localidade_relacional AS r
ON r.codigo_municipio = d.codigo_municipio

WHEN MATCHED AND (r.nome_municipio <> d.nome_municipio OR r.nome_estado <> d.nome_estado
                  OR r.nome_regiao <> d.nome_regiao) THEN
    UPDATE SET nome_municipio = r.nome_municipio,
               nome_estado    = r.nome_estado,
               nome_regiao    = r.nome_regiao

WHEN NOT MATCHED THEN
    INSERT (codigo_municipio, nome_municipio, nome_estado, nome_regiao)
    VALUES (r.codigo_municipio, r.nome_municipio, r.nome_estado, r.nome_regiao)

In [ ]:
%%sql
SELECT * FROM gold.dim_localidade ORDER BY sk_localidade

## 6. Tabela Fato: `fato_vendas`

Grain: um registro por combinação de (data do pedido, produto, cliente, localidade).

In [ ]:
%%sql
DROP TABLE IF EXISTS gold.fato_vendas

In [ ]:
%%sql
CREATE TABLE gold.fato_vendas (
    FK_TEMPO           DATE,
    FK_PRODUTO         BIGINT,
    FK_CLIENTE         BIGINT,
    FK_LOCALIDADE      BIGINT,
    QT_ITENS           INT,
    VL_TOTAL           DOUBLE,
    VL_DESCONTO        DOUBLE
)
USING DELTA;

In [ ]:
df_item_pedido.createOrReplaceTempView("item_pedido")
df_pedido.createOrReplaceTempView("pedido")
df_endereco.createOrReplaceTempView("endereco")

In [ ]:
%%sql
WITH pedido_endereco AS (
    SELECT p.codigo_pedido,
           p.codigo_cliente,
           p.data_pedido,
           e.codigo_municipio
      FROM pedido p
           INNER JOIN endereco e ON p.codigo_cliente = e.codigo_cliente
)
INSERT INTO gold.fato_vendas
SELECT dtem.data                               AS FK_TEMPO,
       dp.sk_produto                           AS FK_PRODUTO,
       dc.sk_cliente                           AS FK_CLIENTE,
       dl.sk_localidade                        AS FK_LOCALIDADE,
       SUM(ip.quantidade_quantidade)           AS QT_ITENS,
       SUM(ip.valor_unitario * ip.quantidade_quantidade) AS VL_TOTAL,
       SUM(ip.valor_desconto)                  AS VL_DESCONTO
  FROM item_pedido ip
       INNER JOIN pedido_endereco pe  ON ip.codigo_pedido  = pe.codigo_pedido
       INNER JOIN gold.dim_produto dp ON ip.codigo_produto  = dp.codigo_produto
       INNER JOIN gold.dim_cliente dc ON pe.codigo_cliente  = dc.codigo_cliente
       INNER JOIN gold.dim_localidade dl ON pe.codigo_municipio = dl.codigo_municipio
       INNER JOIN gold.dim_tempo dtem  ON pe.data_pedido    = dtem.data
 GROUP BY dtem.data, dp.sk_produto, dc.sk_cliente, dl.sk_localidade

In [ ]:
%%sql
SELECT * FROM gold.fato_vendas ORDER BY fk_tempo, fk_produto

## 7. Consulta analítica de exemplo

Total de vendas por mês e categoria de produto.

In [ ]:
%%sql
SELECT t.NomeMes,
       t.Ano,
       p.NOME_CATEGORIA,
       COUNT(1)           AS qtd_pedidos,
       SUM(f.QT_ITENS)    AS total_itens,
       SUM(f.VL_TOTAL)    AS receita_bruta,
       SUM(f.VL_DESCONTO) AS total_descontos,
       SUM(f.VL_TOTAL - f.VL_DESCONTO) AS receita_liquida
  FROM gold.fato_vendas f
       INNER JOIN gold.dim_tempo      t ON f.FK_TEMPO      = t.Data
       INNER JOIN gold.dim_produto    p ON f.FK_PRODUTO     = p.SK_PRODUTO
 GROUP BY t.NomeMes, t.Ano, t.Mes, p.NOME_CATEGORIA
 ORDER BY t.Ano, t.Mes, p.NOME_CATEGORIA